# Precompute cached embeddings for frozen feature extractors

This notebook computes and stores per-image embeddings for frozen feature extractors, so that they don't have to be recomputed every time.


In [35]:
from src.helpers import *
from src.frozen_pipeline import *

In [36]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CSV_PATH = Path("experiments/experiments_1_36.csv")
TOP_FOLDER = Path("photos/top_view_images")
SIDE_FOLDER = Path("photos/side_view_images")
EMBEDDING_CACHE_DIR = Path("embeddings")
EMBEDDING_CACHE_DIR.mkdir(exist_ok=True)

BACKBONE_NAMES = [
    "resnet50",
    "convnext_tiny",
    "densenet121",
]


In [37]:
def precompute_embeddings(backbone_name, image_paths):
    spec = get_backbone_spec(backbone_name, DEVICE)
    n_new = 0
    n_existing = 0

    for image_path in tqdm(image_paths, desc=f"Precomputing {backbone_name}"):
        cache_path = embedding_cache_path(backbone_name, image_path, EMBEDDING_CACHE_DIR)
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        if cache_path.exists():
            n_existing += 1
            continue
        vector = compute_embedding(image_path, spec)
        np.save(cache_path, vector)
        n_new += 1

    return {"backbone": backbone_name, "existing": n_existing, "new": n_new, "total": len(image_paths)}

In [38]:
samples, image_paths = load_image_paths(CSV_PATH, TOP_FOLDER, SIDE_FOLDER)
display(samples[["exp_id","volume","top_path","side_path"]].sort_values("exp_id").head(7))
print("Samples:", samples.shape)
print("Unique images:", len(image_paths))


,exp_id,volume,top_path,side_path
0,1,38,photos/top_view_images/P2090072.JPG,photos/side_view_images/P2090161.JPG
1,1,57,photos/top_view_images/P2090073.JPG,photos/side_view_images/P2090163.JPG
2,1,76,photos/top_view_images/P2090074.JPG,photos/side_view_images/P2090167.JPG
3,2,19,photos/top_view_images/P2090075.JPG,photos/side_view_images/P2090171.JPG
4,2,38,photos/top_view_images/P2090076.JPG,photos/side_view_images/P2090173.JPG
5,2,57,photos/top_view_images/P2090077.JPG,photos/side_view_images/P2090175.JPG
6,2,76,photos/top_view_images/P2090078.JPG,photos/side_view_images/P2090177.JPG


Samples: (140, 23)
Unique images: 280


In [39]:
rows = []
for backbone_name in BACKBONE_NAMES:
    rows.append(precompute_embeddings(backbone_name, image_paths))
cache_summary = pd.DataFrame(rows)
cache_summary


Precomputing densenet121: 100%|██████████| 280/280 [00:00<00:00, 27343.54it/s]


,backbone,existing,new,total
0,resnet50,280,0,280
1,convnext_tiny,280,0,280
2,densenet121,280,0,280
